#### **Problem Statement**

- Retrieval Augmented Generation is a prompting technique for optimizing the output of a large language model, by - referencing a knowledge source in addition to the training data sources before a response is generated.

- Why is Retrieval Augmented Generation important? : Improves **factual consistency** in response and making the response more trustworthy, addressing the problem of hallucination.

- Without retrieval augmented generation, a large language model takes a user input and provides a response, based on the information the model has been trained on. With retrieval augmented generation, an information retrieval component is combined with the text generation model.

- The large language model hence uses the external data source combined with the information the model has been trained on to provide a better response.

#### **References**

- https://www.promptingguide.ai/techniques/rag

- Lewis, Patrick, et al. "Retrieval-augmented generation for knowledge-intensive nlp tasks." Advances in Neural Information Processing Systems 33 (2020): 9459-9474.

- https://aws.amazon.com/what-is/retrieval-augmented-generation/

- https://www.kaggle.com/datasets/radek1/sci-or-not-sci-hypthesis-testing-pack

- https://www.neum.ai/post/

#### **Installing Dependencies**

In [ ]:
# Installing dependencies : Langchain - Framework, ChromaDB (vector store) and tiktoken for use with openAI model

# Pinned versions (tested Oct 2026) so this notebook keeps working on Google Colab
!pip install -q langchain==0.3.27 langchain-community==0.3.27 langchain-core==0.3.72 langchain-text-splitters==0.3.9 langchain-openai==0.3.28 openai==1.99.9 tiktoken==0.14.0 chromadb==0.5.23
# !pip install langchain -q   # (replaced by the pinned install above)
# !pip install langchain-openai -q   # (replaced by the pinned install above)
# !pip install chromadb -q   # (replaced by the pinned install above)
# !pip install tiktoken -q   # (replaced by the pinned install above)

#### **Importing libraries**

In [ ]:
# Importing required dependencies :
import csv
from google.colab import userdata # Importing the OpenAI API KEY in Colab Environment
from langchain.docstore.document import Document #Langchain Document Object
from langchain.text_splitter import CharacterTextSplitter #Text splitter
from langchain_openai import ChatOpenAI as OpenAI  # completion model retired; chat model used instead  #Importing OpenAI Interface
from langchain_openai import OpenAIEmbeddings #OpenAI Embeddings
from langchain_community.vectorstores import Chroma # Importing Chroma vector database for storing emdeddings
from langchain.chains import ConversationalRetrievalChain # Importing ConversationalRetrievalChain for Q&A

#### **CSV file for QA**

In [ ]:
# Login to Google Account
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Extract specific columns : prompt, wikipedia_excerpt from the csv file  - data_QA_.csv
# Create a list consisting of each row of the csv file for further processing.
# https://www.neum.ai/post/
# https://python.langchain.com/docs/integrations/document_loaders

columns = ['prompt','wikipedia_excerpt']
docs = []
with open('/content/drive/MyDrive/data_QA_.csv', encoding = 'utf-8') as csvfile:
  csv_reader = csv.DictReader(csvfile) #csv.DictReader() method creates an object like a regular reader, maps the information in each row to a dictionary
  for i, row in enumerate(csv_reader):
    value = {k: row[k] for k in columns if k in row}
    to_embed = "\n".join(f"{k.strip()}: {v.strip()}" for k, v in value.items())
    new_Doc = Document(page_content = to_embed) #Create a langchain Document Object
    docs.append(new_Doc)

In [ ]:
type(new_Doc)

langchain_core.documents.base.Document

#### **Text splitter**

- Splits text based on characters and measure chunk length by number of characters to fit into the model's context window.  

- https://python.langchain.com/docs/modules/data_connection/document_transformers/character_text_splitter

- https://python.langchain.com/docs/modules/data_connection/document_transformers/

In [ ]:
# Create a CharacterTextSplitter() with chunk_size = 500 and chunk_overlap = 20
# creating meaningful chunks with some overlap for ensuring context between chunks.

text_splitter = CharacterTextSplitter(
    separator = '\n',
    chunk_size=500,
    chunk_overlap = 20,
    length_function = len,
)
texts = text_splitter.split_documents(docs)

#### **Embeddings**

- Once the data is broken into meaningful chunks, these need to be converted into a vector representation, which involves transforming the text data into embeddings which are numeric representations that capture the semantic meaning behind text.

- Examples : OpenAI Embeddings, HuggingFaceEmbeddings with specific models such as BERT.

- https://www.datacamp.com/blog/what-is-retrieval-augmented-generation-rag

In [ ]:
# Extracting the OpenAI API key using userdata() method from google.colab, Initializing OpenAIEmbeddings() class

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
embeddings_model = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

#### **Vector Store**

- Chroma vector database to store the embedded text vector
- Other examples : FAISS, Weaviate vector database
- https://python.langchain.com/docs/modules/data_connection/vectorstores.html

In [ ]:
# Chroma.from_documents() method takes in the text chunks with an appropriate model for embeddings
db = Chroma.from_documents(texts, embeddings_model)

#### **Retriever interface**

- VectorStoreRetriever, backed by a VectorStore (Chroma)
- When a query enters a system, it must also be converted into embeddings. The same model must be used for document and query embedding to ensure uniformity between the two.
- Two search types to retrieve text : Retrieve text based on similarity : uses similarity search in the retriever object, where it selects the text chunk vectors that are most similar (using cosine similarity or Euclidean distance) to the question vector.
- "MMR", maximum marginal relevance search, optimizing for similarity to query AND diversity among the selected documents.

- https://towardsdatascience.com/4-ways-of-question-answering-in-langchain-188c6707cc5a

In [ ]:
# Vector store retriever
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":5})

#### **Creating a chain for QA**

- Uses load_qa_chain, provides a generic interface for answering questions.

- Retrieval QA Chain, retrieves the most relevant chunk of text and feeds those to the language model.

- Conversational Retrieval Chain takes chat history for follow up questions.

- ConversationalRetrievalChain = RetrievalQAChain + conversation Memory

- https://api.python.langchain.com/en/latest/chains/langchain.chains.conversational_retrieval.base.ConversationalRetrievalChain.html

In [ ]:
# gpt-4o-mini model for the ConversationalRetrievalChain
llm = OpenAI(openai_api_key = OPENAI_API_KEY, model_name = "gpt-4o-mini")
chain = ConversationalRetrievalChain.from_llm(llm = llm, retriever = retriever)
chat_history = []

In [ ]:
# Query for the large language model :
query = "How much rainfall did Kelambakkam, Chennai receive on 19 May during Cyclone Roanu?"

In [ ]:
response = chain({"question" : query, "chat_history": chat_history})

In [ ]:
# Extracting response from the model
response['answer']

' Kelambakkam, Chennai received 226 mm of rainfall on 19 May during Cyclone Roanu.'